# 08 · Visualisasi untuk Bab IV
Semua grafik dibuat dari tabel dan riwayat yang disimpan notebook sebelumnya, lalu disimpan 300 dpi di `outputs/<kategori>/figures/`
(syarat resolusi gambar JuTISI).

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
import matplotlib.pyplot as plt
from hr import plots, experiment as X
T = lambda n: cfg.path("tables", n)
F = lambda n: cfg.path("figures", n)
MS, TG = cfg.main_scale, cfg.main_target

## 1. Kurva loss dan NDCG@10 validasi per epoch
Dipakai untuk menunjukkan proses pelatihan, *early stopping*, dan ada tidaknya *overfitting*.

In [ ]:
codes = [c for c in ["U", "B5", "B4"] if (cfg.out_dir / "runs" / X.run_id(c, MS, TG, cfg.seeds[0])).exists()]
fig, axes = plt.subplots(len(codes), 2, figsize=(9, 3 * len(codes)), squeeze=False)
for row, c in zip(axes, codes):
    h = pd.read_csv(cfg.out_dir / "runs" / X.run_id(c, MS, TG, cfg.seeds[0]) / "history.csv")
    plots.loss_curves(h, axes=row); row[0].set_title(c)
fig.tight_layout(); plots.save(fig, F("training_loss_ndcg.png"))

## 2. Perbandingan baseline (RM1)

In [ ]:
ax = plots.baseline_figure(pd.read_csv(T("uji_baseline.csv")), pd.read_csv(T("uji_acak.csv")))
plots.save(ax.figure, F("uji_baseline.png"))

## 3. Ablasi dan skala data (RM2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
a = pd.read_csv(T("uji_ablasi.csv")).rename(columns={"ndcg@10": "mean", "ndcg@10_lo": "lo", "ndcg@10_hi": "hi"})
plots.bar_ci(a, "code", "NDCG@10 uji", ax=axes[0]); axes[0].set_title("Ablasi fitur")
s = pd.read_csv(T("uji_skala.csv")).rename(columns={"ndcg@10": "mean", "ndcg@10_lo": "lo", "ndcg@10_hi": "hi"})
plots.scale_curve(s, ax=axes[1])
fig.tight_layout(); plots.save(fig, F("uji_ablasi_skala.png"))

## 4. Usia ulasan (RM3)

In [ ]:
fig = plots.age_figure(pd.read_csv(T("age_kurva.csv")), pd.read_csv(T("uji_per_kelompok_usia.csv")))
plots.save(fig, F("usia.png"))

## 5. Ukuran produk

In [ ]:
ax = plots.size_figure(pd.read_csv(T("uji_per_ukuran_produk.csv")))
plots.save(ax.figure, F("uji_ukuran_produk.png"))
print(sorted(p.name for p in (cfg.out_dir / "figures").glob("*.png")))

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()